*Credits: Aleksandar Marinkovic.*

# Horizontal Banding Removal

V4 Miniscope frames transiently carry a banding artifact: a cluster of
consecutive sensor rows that is uniformly brighter or darker than its neighbours for a
single frame. It survives spatial band-pass filtering and can leak into CNMF-E components, so it needs to be removed.

In [ ]:
# Setup: find the repo root, put analysis/src on sys.path, then pull in every
# package/constant from utils/constants_and_packages.py (imports live there).
import sys
from pathlib import Path

_REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_REPO_ROOT / "analysis" / "src"))

from utils.constants_and_packages import *

add_src_to_path()  # every analysis/src/<area> dir, so analysis modules import by name

from utils.analysis_utils import get_all_mice, get_all_rounds, get_miniscope_videos, setup_style
import banding_preprocessing as bp

setup_style()


## Config

Set `-1` to include **all** available values for that field:
- `MOUSE = -1` — all mice in `data/`
- `ROUND = -1` — all rounds for each selected mouse
- `DAY   = -1` — all sessions for each selected mouse/round

`WIN`, `Z`, `DILATE` and `POLARITY` each take a single value **or a list**. Lists are
expanded into every combination and each combination is processed and written to its own
folder, so a sweep is one run. All-scalar settings give a single combination, which is the
ordinary case.

`WIN` must comfortably exceed the thickness of a stripe cluster, or the median baseline is
computed *from* banded rows and the artefact is only partly removed. On Mouse944 a cluster
spans ~50 rows: `WIN = 9` leaves about 70 % of the artefact behind, `WIN = 31` leaves 3 %,
and `WIN = 51` adds nothing further.

`Z` is set from the elbow in section 2. Detection reads every frame, so `SUBSAMPLE` stays
at 1 — a stripe lasts one frame, and sub-sampling would both miss events and leave the
residuals unusable for correction.

In [ ]:
MOUSE    = 944        # int or -1 for all
TRACK    = "Linear"
ROUND    = 1          # int or -1 for all
DAY      = 0          # 0-based int or -1 for all

# Each of the four below takes a scalar or a list; lists give every combination.
WIN      = [61, 101, 151]        # rows; median-filter width of the row-profile baseline
Z        = [6.0, 8.0, 10.0]        # robust z threshold for calling a row banded
DILATE   = 3          # rows; dilation of the flagged mask along y
POLARITY = "both"     # "bright", "dark", or "both"
SUBSAMPLE = 1         # keep at 1; correction needs every frame

SAVE_MASK   = True    # write band_mask.npz for every parameter combination
WRITE_VIDEO = True    # write the corrected AVI (a few minutes and ~1 GB per combination)
CODEC       = "MPEG4"  # "FFV1" (lossless, for analysis) or "MPEG4" (lossy, previews only)

PARAMS = bp.param_grid(win=WIN, z=Z, dilate=DILATE, polarity=POLARITY)
print(f"{len(PARAMS)} parameter combination(s):")
for p in PARAMS:
    print("  " + bp.param_tag(p))

## Resolve Sessions

In [ ]:
mice = get_all_mice() if MOUSE == -1 else [MOUSE]

SESSIONS = []
for m in mice:
    rounds = get_all_rounds(m, TRACK) if ROUND == -1 else [ROUND]
    for r in rounds:
        try:
            paths = get_miniscope_videos(m, TRACK, round=r)
        except FileNotFoundError as e:
            print(f"Skipping Mouse{m} Round{r}: {e}")
            continue
        days = list(range(len(paths))) if DAY == -1 else [DAY]
        for d in days:
            if 0 <= d < len(paths):
                round_str = f"Round{r}" if r is not None else "(no round)"
                label = f"Mouse{m}  |  {round_str}  |  Day {d + 1}"
                SESSIONS.append(dict(mouse=m, round=r, day=d,
                                     path=paths[d], label=label))

print(f"{len(SESSIONS)} session(s) queued:")
for s in SESSIONS:
    print(f"  {s['label']}")
    print(f"    {s['path']}")

## 1. Row Profiles

Each frame is reduced to its (H,) row-mean profile while streaming, so a full-length
recording costs a few MB instead of the ~22 GB the movie itself would need.

In [ ]:
for s in SESSIONS:
    display(Markdown(f"### {s['label']}"))
    s['rows'], s['meta'] = bp.compute_row_profiles(s['path'], subsample=SUBSAMPLE)
    print(f"Row profiles: {s['rows'].shape}  "
          f"({s['meta']['n_frames_used']} frames x {s['meta']['height']} rows)")

## 2. Choosing the Threshold

**Left:** distribution of the per-row detection statistic on a log count axis. Noise decays
steeply near zero; banded rows form the flat heavy tail. **Right:** how many frames each
candidate threshold flags.

Pick `Z` where the sensitivity curve flattens — below the elbow the count explodes because
ordinary row noise starts being flagged, above it the same discrete stripe events are found
regardless of threshold.

In [ ]:
for s in SESSIONS:
    display(Markdown(f"### {s['label']}  —  {bp.param_tag(PARAMS[0])}"))
    res = bp.detect_bands(s['rows'], **PARAMS[0])
    display(bp.plot_score_distribution(res)['table'])

## 2b. Parameter Sweep

One row per combination in `PARAMS`, so `WIN` and `Z` can be compared against each other
rather than one at a time. The row profiles from section 1 are reused, so a combination
costs only the median filter — a few seconds — and nothing is written here.

Read it as a stability check: a combination sitting on a plateau, where neighbouring `Z`
values flag nearly the same number of frames, is a safe choice. `pct_rows` climbing while
`rows_per_bad_frame` stays flat means new *events* are being found; both climbing together
means ordinary row noise is starting to be flagged.

In [ ]:
for s in SESSIONS:
    display(Markdown(f"### {s['label']}"))
    s['sweep'] = bp.sweep_bands(s['rows'], params=PARAMS)
    display(s['sweep'])

## 3. Detection

**Top:** the detection statistic for every (frame, row). A stripe event is a short bright
vertical dash — one frame wide, localised in y. The faint horizontal ridge across the middle
rows is not banding: it is ordinary row noise scaling with intensity in the brightest part of
the vignette, and it stays well below threshold.

**Bottom:** the per-frame peak, with flagged frames marked.

In [ ]:
for s in SESSIONS:
    for p in PARAMS:
        display(Markdown(f"### {s['label']}  —  {bp.param_tag(p)}"))
        res = bp.detect_bands(s['rows'], **p)
        bp.print_band_summary(res, s['meta'])
        bp.plot_band_map(res, s['meta'], label=f"{s['label']} — {bp.param_tag(p)}")

## 4. Row Profile of the Worst Frame

The baseline (orange) should track the vignette smoothly and ignore the band. If it bends
into the spike instead, `WIN` is too narrow for this stripe cluster and the correction in
section 5 will leave a visible remnant.

In [ ]:
for s in SESSIONS:
    for p in PARAMS:
        display(Markdown(f"### {s['label']}  —  {bp.param_tag(p)}"))
        res = bp.detect_bands(s['rows'], **p)
        if res['bad'].any():
            bp.plot_row_profile(res, int(res['trace'].argmax()), s['meta'])
        else:
            print("No banded frames detected.")

## 5. Before / After

Each flagged row has its residual subtracted from every pixel in that row, restoring the row
mean to the local baseline while leaving neuron signal — which varies along x — untouched.

The **Removed** panel is the acceptance test: it must contain horizontal lines and nothing
else. Any soma, vignette, or diffuse background visible there means real signal is being
taken out.

In [ ]:
for s in SESSIONS:
    for p in PARAMS:
        display(Markdown(f"### {s['label']}  —  {bp.param_tag(p)}"))
        res = bp.detect_bands(s['rows'], **p)
        bp.plot_worst_frames(s['path'], res, s['meta'], n=2)

## 6. Save the Band Mask

Everything this notebook writes goes to `<session_dir>/denoised/MMDDYYYY-HHMMSS/<tag>/`,
where the timestamp is the moment this run started and `<tag>` names the parameter
combination (`win101_z8_dil3_both`). One timestamp covers the whole sweep, so its
combinations sit side by side and an earlier run is never overwritten.

`band_mask.npz` (keys `bad`, `resid`, `score`) records which rows were corrected and by how
much, so later steps can exclude them or audit the correction without re-running detection.

In [ ]:
RUN_STAMP = datetime.now().strftime(bp.RUN_STAMP_FMT)   # MMDDYYYY-HHMMSS

for s in SESSIONS:
    s['run_dirs'] = {bp.param_tag(p): bp.make_run_dir(s['path'], stamp=RUN_STAMP,
                                                      tag=bp.param_tag(p))
                     for p in PARAMS}
    for d in s['run_dirs'].values():
        print(d)

if SAVE_MASK:
    for s in SESSIONS:
        for p in PARAMS:
            res = bp.detect_bands(s['rows'], **p)
            bp.save_band_mask(res, out_dir=s['run_dirs'][bp.param_tag(p)])
else:
    print("SAVE_MASK is False; no mask written.")

## 7. Write the Corrected Video

Streams the source, corrects each frame, and writes 8-bit greyscale to that combination's
folder from section 6: `<session_dir>/denoised/MMDDYYYY-HHMMSS/<tag>/debanded_<name>.avi`.

`CODEC` picks the encoding. **FFV1 (lossless)** is the default on purpose: the artefact is
only a few grey levels deep in most frames, and a lossy re-encode would be the same order of
magnitude as the thing being removed. **MPEG4** (`FMP4`, the codec the Miniscope itself
records in) writes files several times smaller and faster — use it when the output is only
being watched, never for anything that feeds CNMF-E.

Detection is recomputed here rather than carried over from the earlier sections: a full
result is ~0.2 GB, so caching one per combination would exhaust memory on a sweep, while
recomputing costs a few seconds against the minutes each video takes to write.

The source video is never modified. Feed the debanded copy to motion correction.

In [ ]:
if WRITE_VIDEO:
    for s in SESSIONS:
        s['out_paths'] = {}
        for p in PARAMS:
            tag = bp.param_tag(p)
            display(Markdown(f"### {s['label']}  —  {tag}"))
            res = bp.detect_bands(s['rows'], **p)
            s['out_paths'][tag] = bp.destripe_video(s['path'], res,
                                                    out_dir=s['run_dirs'][tag],
                                                    codec=CODEC)
else:
    print("WRITE_VIDEO is False; nothing written.")